# GMSH Mesh Generation via MeepMeshSim

This notebook demonstrates using `MeepMeshSim` to generate a GMSH mesh
from any gdsfactory component + LayerStack — mirroring Palace's
`mesh()` → `write_config()` workflow for photonic simulations.

We use a UBC PDK Y-branch (photonic SOI) as an example.

### Load component from UBC PDK

In [ ]:
from ubcpdk import PDK, cells

PDK.activate()

# c = cells.ebeam_y_1550()
# c = cells.ring_single()
c = cells.coupler()

dirname = "./sim-data-mesh-----coupler"

c

### Configure MeepMeshSim

In [ ]:
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.meep import MeepMeshSim

# Build a LayerStack for the SOI process
stack = LayerStack(pdk_name="ubcpdk")

stack.layers["core"] = Layer(
    name="core",
    gds_layer=(1, 0),
    zmin=0.0,
    zmax=0.22,
    thickness=0.22,
    material="si",
    layer_type="dielectric",
)

stack.dielectrics = [
    {"name": "box", "material": "SiO2", "zmin": -3.0, "zmax": 0.0},
    {"name": "clad", "material": "SiO2", "zmin": 0.0, "zmax": 1.8},
]

stack.materials = {
    "si": {"type": "dielectric", "permittivity": 11.7},
    "SiO2": {"type": "dielectric", "permittivity": 2.1},
}

# Configure simulation
sim = MeepMeshSim()
sim.geometry(component=c, stack=stack)
sim.materials = {"si": 3.47, "SiO2": 1.44}
sim.source(port="o1", wavelength=1.55, wavelength_span=0.1, num_freqs=11)
sim.monitors = ["o1", "o2", "o3"]
sim.domain(pml=1.0, margin=0.5)
sim.solver(resolution=32)

### Generate mesh and write config

In [ ]:
result = sim.mesh(dirname, refined_mesh_size=0.2, max_mesh_size=1.0)
config_path = sim.write_config()

print(f"Mesh file: {result.mesh_path}")
print(f"File size: {result.mesh_path.stat().st_size / 1024:.0f} KB")
print(f"Config: {config_path}")

### Mesh summary

In [ ]:
import meshio
import numpy as np

stats = result.mesh_stats
groups = result.groups
m = meshio.read(result.mesh_path)
tag_to_name = {tag: name for name, (tag, _) in m.field_data.items()}

# --- Header ---
size_kb = result.mesh_path.stat().st_size / 1024
print(f"Mesh: {result.mesh_path.name}  ({size_kb:.0f} KB)")
print(f"Nodes: {stats.get('nodes', '?'):,}   Tets: {stats.get('tetrahedra', '?'):,}")
print()

# --- Quality ---
q = stats.get("quality", {})
sicn = stats.get("sicn", {})
edge = stats.get("edge_length", {})
print("Quality")
print(
    f"  Shape (gamma):  min={q.get('min', '?')}  mean={q.get('mean', '?')}  max={q.get('max', '?')}"
)
print(
    f"  SICN:           min={sicn.get('min', '?')}  mean={sicn.get('mean', '?')}  invalid={sicn.get('invalid', '?')}"
)
print(f"  Edge length:    min={edge.get('min', '?')}  max={edge.get('max', '?')}")

# Edge ratio from actual mesh
for cells in m.cells:
    if cells.type == "tetra":
        pts = m.points[cells.data]
        pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
        all_edges = np.stack(
            [np.linalg.norm(pts[:, i] - pts[:, j], axis=1) for i, j in pairs]
        )
        ratios = all_edges.max(axis=0) / np.maximum(all_edges.min(axis=0), 1e-15)
        bad = int(np.sum(ratios > 20))
        print(
            f"  Edge ratio:     mean={ratios.mean():.1f}  max={ratios.max():.1f}  bad(>20)={bad}"
        )
print()

# --- Bounding box ---
bb = stats.get("bbox", {})
print(f"Bounding box")
print(
    f"  x: [{bb.get('xmin', 0):.2f}, {bb.get('xmax', 0):.2f}]  ({bb.get('xmax', 0) - bb.get('xmin', 0):.2f} um)"
)
print(
    f"  y: [{bb.get('ymin', 0):.2f}, {bb.get('ymax', 0):.2f}]  ({bb.get('ymax', 0) - bb.get('ymin', 0):.2f} um)"
)
print(
    f"  z: [{bb.get('zmin', 0):.2f}, {bb.get('zmax', 0):.2f}]  ({bb.get('zmax', 0) - bb.get('zmin', 0):.2f} um)"
)
print()

# --- Physical groups ---
print("Physical groups")
for cells, phys in zip(m.cells, m.cell_data["gmsh:physical"]):
    if cells.type != "tetra":
        continue
    for tag, name in sorted(tag_to_name.items(), key=lambda x: x[1]):
        mask = phys == tag
        n = int(np.sum(mask))
        if n == 0:
            continue
        pts = m.points[cells.data[mask].ravel()]
        zlo, zhi = pts[:, 2].min(), pts[:, 2].max()
        mat = ""
        if name in groups.get("layer_volumes", {}):
            mat = f"  (material: {groups['layer_volumes'][name].get('material', '?')})"
        print(f"  {name:16s}  {n:>7,} tets   z=[{zlo:.3f}, {zhi:.3f}]{mat}")

if groups.get("port_surfaces"):
    print()
    print("Port surfaces")
    for name, info in groups["port_surfaces"].items():
        c = info["center"]
        print(
            f"  {name:16s}  center=({c[0]:.2f}, {c[1]:.2f})  width={info['width']:.2f}  z=[{info['z_range'][0]:.3f}, {info['z_range'][1]:.3f}]"
        )

### Visualize the mesh

In [ ]:
# Full mesh wireframe
sim.plot_mesh(interactive=True)

In [ ]:
# Show only the waveguide core
sim.plot_mesh(show_groups=["core", "port_"], interactive=False)